In [2]:
# 4. How does the historical success over the last 30-40 years of the 10 most popular active directors
# influence the financial and critical success of their new movie releases?

import os
import requests
from datetime import datetime
from dotenv import load_dotenv
import time
import numpy as np

load_dotenv('../../test.env')
api_key_TMDB = os.getenv("TMDB_API_KEY")
api_key_OMDB = os.getenv("OMDB_API_KEY1")

if not api_key_OMDB:
    print("OMDB API KEY not found check env file")
    exit()
else:
    print("OMDB KEY loaded")

if not api_key_TMDB:
    print("TMDB API KEY not found check env file")
    exit()
else:
    print("TMDB KEY loaded")




OMDB KEY loaded
TMDB KEY loaded


In [3]:
#Building our links for the different api
tmdb_base = "https://api.themoviedb.org/3"
tmbd_person_endpoint = "/person/popular"  #Get all result for "" fitting criteria
tmdb_person_url = tmdb_base + tmbd_person_endpoint

omdb_base = f"http://www.omdbapi.com/?"


In [4]:
#Final list of the 10 most popular directors
dir_result = []
total_pages = 1

parameters = {
    'api_key' : api_key_TMDB,
    'page' : 1
}
i = 0 # Person index
while len(dir_result) < 10 and parameters['page'] <= total_pages:
    
    # Getting a list of people, listed on TMDb
    person_response = requests.get(tmdb_person_url, params=parameters)
    
    if person_response.status_code == 200:
        person_data = person_response.json()
        total_pages = person_data['total_pages']
        
        
        # Going through list of all people listed on TMDb
        for person in person_data['results']:
            
             
            # Resetting values for each person
            avg_director_score = 0
            newest_movie_release_year = 0000
            total_vote_count = 0
            j = 0 # Index for movies
            directed_movies = []

            # Checking if current person is Actor or Director
            if person['known_for_department'] == 'Directing':
                person_id = person['id']
                
                # request link to get details about each director, to check if still active and one of the most popular directors
                tmdb_credit_url = f"https://api.themoviedb.org/3/person/{person['id']}/movie_credits"
                credit_params = {
                    'api_key' : api_key_TMDB
                }
                
                try:
                    credit_response = requests.get(tmdb_credit_url, params=credit_params)
                    if credit_response.status_code == 200:
                        credit_data = credit_response.json()
                except:
                    pass

                # Filtering out niche Directors, that directed less than 5 movies
                if len(credit_data['crew'])>=5:
                    
                    # Going through movie list, that the director is credited in as crew member
                    for dir_movie in credit_data['crew']:
                        score_list=[]
                        imdb_score = 0.0
                        meta_score = 0.0
                        rotten_score = 0.0

                        # Only looking at movies, where person's job was directing 
                        if dir_movie['job'] == 'Director':


                            # Filtering out niche movies, that don't have a certain vote count and that don't have a budget listed
                            if dir_movie['vote_count']>500:
                                j += 1 
                                # Counting the amount of popular movies to calculate average score of all movies

                                movie_release_year = int(dir_movie['release_date'][0:4])

                                # Updating the newest movie release, to check, if director still active
                                if movie_release_year > newest_movie_release_year:
                                    newest_movie_release_year = movie_release_year

                                # Getting details about movie, for IMDb-ID 
                                tmdb_movie_url = f"https://api.themoviedb.org/3/movie/{dir_movie['id']}"
                                tmdb_movie_params= {
                                    'api_key' : api_key_TMDB
                                }

                                try:
                                    tmdb_movie_response = requests.get(tmdb_movie_url, params=tmdb_movie_params)
                                    if tmdb_movie_response.status_code == 200:
                                        tmdb_movie_data = tmdb_movie_response.json()
                                except:
                                    pass
                                
                                # Filtering niche movies, by only looking at bigger productions that have their budget recorded
                                if tmdb_movie_data['budget'] > 0:

                                    omdb_params = {
                                        'apikey' : api_key_OMDB,
                                        'i' : tmdb_movie_data['imdb_id'] # Using IMDb-ID, to get IMDb, Metacritic and Rotten Tomatoes score for movie
                                    }
                                    try:
                                        omdb_response = requests.get(omdb_base, params=omdb_params)
                                        if omdb_response.status_code == 200:
                                            omdb_data = omdb_response.json()
                                    except:
                                        pass
                                    
                                    # Getting a total sum of Votes on IMDb to asses the popularity of a director
                                    imdb_vote_count = omdb_data['imdbVotes']
                                    imdb_vote_count = int(imdb_vote_count.replace(',',''))
                                    total_vote_count += imdb_vote_count

                                    # Gathering TMDb, IMDb, Metacritic and Rotten Tomatoes scores and calculating their mean, to get average movie score
                                    try:
                                        tmdb_score = tmdb_movie_data['vote_average']
                                        score_list.append(tmdb_score)
                                    except:
                                        pass

                                    try:
                                        rotten_score = omdb_data['Ratings'][1]['Value']

                                        rotten_score = rotten_score[0:-1] # removing "%" from the end of the string to be able to convert string to float

                                        rotten_score = ((float(rotten_score))/10)
                                        score_list.append(rotten_score)
                                    except:
                                        pass

                                    try:
                                        meta_score = ((float(omdb_data['Metascore']))/10) # converting Metascore x/100 to decimal number x/10
                                        score_list.append(meta_score)
                                    except:
                                        pass

                                    try:
                                        imdb_score = float(omdb_data['imdbRating'])
                                        score_list.append(imdb_score)
                                    except:
                                        pass

                                    avg_movie_score = sum(score_list)/len(score_list)
                                    
                                    # Collecting sum of all average movie scores, to get overall score for director
                                    avg_director_score += avg_movie_score
                                    
                                    # Adding movie to the list of qualified movies to be looked at per director
                                    directed_movies.append({
                                        'movie_index' : j,
                                        'director_index' : i,
                                        'title' : dir_movie['title'],
                                        'budget' : tmdb_movie_data['budget'],
                                        'revenue' : tmdb_movie_data['revenue'],
                                        'release_year' : movie_release_year,
                                        'avg_score' : round(avg_movie_score, 2),
                                        'IMDb_Vote_count' : imdb_vote_count

                                    })

                                    # Spam proctection
                                    time.sleep(0.25)
                                
                                # Spam proctection
                                time.sleep(0.25)


                    # Filtering directors with less than 3 popular movies
                    if len(directed_movies) >= 3:

                        # Calculating average director score and log of total IMDb vote count
                        avg_director_score = avg_director_score/len(directed_movies)
                        log_votes = np.log(total_vote_count)

                        print(person['name'])

                        # Filtering directors by score, amount of IMDb Votes and if they're still active
                        if (
                            avg_director_score > 7
                            and newest_movie_release_year > 2019
                            and log_votes >=10.5
                        ):
                            
                            # request link to get details about directors, to get IMDb id
                            dir_detail_url = f"https://api.themoviedb.org/3/person/{person['id']}"
                            dir_detail_params = {
                                'api_key' : api_key_TMDB
                            }
                
                            try:
                                dir_detail_response = requests.get(dir_detail_url, params=dir_detail_params)
                                if dir_detail_response.status_code == 200:
                                    dir_detail_data = dir_detail_response.json()
                            except:
                                pass

                            person_dict = {
                                'index' : i,
                                'name' : person['name'],
                                'avg_score' : round(avg_director_score,2),
                                'total_vote_count' : total_vote_count,
                                'latest_movie_release_year' : newest_movie_release_year,
                                'tmdb_id' : person['id'],
                                'imdb_id' : dir_detail_data['imdb_id'],

                            }
                            dir_result.append((person_dict, directed_movies))
                            print('collected director: '+person_dict['name'])
                            i += 1


        print(f"Fetched page {parameters['page']}")
        parameters['page']+=1

        #Spam protection
        time.sleep(0.25)
                
    else:
        print(f"Error on page {parameters['page']}")
        break

print(dir_result)

Fetched page 1
Fetched page 2
Fetched page 3
Fetched page 4
Ryan Coogler
collected director: Ryan Coogler
Fetched page 5
Fetched page 6
Steven Spielberg
collected director: Steven Spielberg
Fetched page 7
Guillermo del Toro
collected director: Guillermo del Toro
Fetched page 8
Christopher Nolan
collected director: Christopher Nolan
Fetched page 9
Fetched page 10
Fetched page 11
Paul Thomas Anderson
collected director: Paul Thomas Anderson
Fetched page 12
Fetched page 13
Martin Scorsese
collected director: Martin Scorsese
Fetched page 14
Fetched page 15
Quentin Tarantino
Fetched page 16
Taika Waititi
collected director: Taika Waititi
Fetched page 17
Wes Anderson
collected director: Wes Anderson
Fetched page 18
Guy Ritchie
Fetched page 19
Fetched page 20
Fetched page 21
James Cameron
collected director: James Cameron
Fetched page 22
Fetched page 23
Fetched page 24
Fetched page 25
Fetched page 26
Hayao Miyazaki
collected director: Hayao Miyazaki
Fetched page 27
[({'index': 0, 'name': 'Rya

In [5]:
import csv

filename = 'director_list.csv'
print("write director data in CSV")

# give writing perm and add utf-8 for "Sonderzeichen"
with open(filename, mode='w', newline='', encoding='utf-8') as file:
    
    # define lines
    columns = ['id', 'name', 'avg_score', 'total_vote_count', 'latest_movie_release_year', 'tmdb_id', 'imdb_id', 'directed_movie_count']
    writer = csv.DictWriter(file, fieldnames=columns)

    # write headline
    writer.writeheader()

    # add each row
    for index, person in enumerate(dir_result, start=0):
        writer.writerow({
            'id': index,
            'name' : person[0]['name'],
            'avg_score' : person[0]['avg_score'],
            'total_vote_count' : person[0]['total_vote_count'],
            'latest_movie_release_year' : person[0]['latest_movie_release_year'],
            'tmdb_id' : person[0]['tmdb_id'],
            'imdb_id' : person[0]['imdb_id'],
            'directed_movie_count' : len(person[1])
        })

print(f"Finished writing '{filename}'")

for person in dir_result:
    filenames = f"director_{person[0]['index']}.csv"
    print('write movie data in CSV')
    
    with open(filenames, mode='w', newline='', encoding='utf-8') as file:

        columns = ['id', 'director_id', 'title', 'budget', 'revenue', 'release_year', 'avg_score', 'imdb_vote_count']
        writer = csv.DictWriter(file, fieldnames=columns)

        writer.writeheader()

        for index, movie in enumerate(person[1], start=0):
            writer.writerow({
                'id' : index,
                'director_id' : movie['director_index'],
                'title' : movie['title'],
                'budget' : movie['budget'],
                'revenue' : movie['revenue'],
                'release_year' : movie['release_year'],
                'avg_score' : movie['avg_score'],
                'imdb_vote_count' : movie['IMDb_Vote_count']
            })

    print(f"finished writing '{filenames}'")


    

write director data in CSV
Finished writing 'director_list.csv'
write movie data in CSV
finished writing 'director_0.csv'
write movie data in CSV
finished writing 'director_1.csv'
write movie data in CSV
finished writing 'director_2.csv'
write movie data in CSV
finished writing 'director_3.csv'
write movie data in CSV
finished writing 'director_4.csv'
write movie data in CSV
finished writing 'director_5.csv'
write movie data in CSV
finished writing 'director_6.csv'
write movie data in CSV
finished writing 'director_7.csv'
write movie data in CSV
finished writing 'director_8.csv'
write movie data in CSV
finished writing 'director_9.csv'
